# 01 Macro Data Collection

This notebook downloads a small set of additional market factors that may help interpret cryptocurrency market behavior in later project stages.

| Macro Factor | Meaning          |
| --- |------------------|
| **DXY** | US dollar index  |
| **VIX** | Volatility index |
| **Gold** | Gold futures (safe haven) |
| **SP500** | S&P 500 index (risk appetite) |

These series are downloaded, cleaned, aligned to the crypto dataset date range, and saved for later use in regime interpretation and possible model extensions.

In [1]:
from pathlib import Path
from typing import Dict

import pandas as pd
import yfinance as yf

## Configuration

This section defines the project paths, selected macro tickers, and output locations.

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MACRO_TICKERS: Dict[str, str] = {
    "DXY": "DX-Y.NYB",
    "VIX": "^VIX",
    "Gold": "GC=F",
    "SP500": "^GSPC",
}

START_DATE = "2017-01-01"
END_DATE = None
INTERVAL = "1d"

# Reference dataset for alignment
CRYPTO_ALIGNED_CLOSE_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_aligned.csv"

# Raw macro dataset
MACRO_RAW_OUTPUT_PATH = DATA_RAW_DIR / "macro_long_raw.csv"

# Clean macro datasets
MACRO_WIDE_FULL_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_full.csv"
MACRO_WIDE_ALIGNED_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_aligned.csv"

# Macro datasets
MACRO_QUALITY_SUMMARY_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_data_quality_summary.csv"

## Download raw macro data

The following function downloads daily market data for a single macro factor and returns it in a consistent tabular structure.

In [3]:
def download_single_series(
    ticker: str,
    symbol: str,
    start_date: str,
    end_date: str | None,
    interval: str,
    ) -> pd.DataFrame:
    """
    Download daily market data for a single ticker from yfinance.

    Parameters
    ----------
    ticker : str
        Internal ticker name used in the project, e.g. DXY.
    symbol : str
        Yahoo Finance symbol.
    start_date : str
        Download start date in YYYY-MM-DD format.
    end_date : str | None
        Download end date in YYYY-MM-DD format. If None, the latest available date is used.
    interval : str
        Data interval, e.g. '1d'.

    Returns
    -------
    pd.DataFrame
        Raw market data with one row per date.
    """
    df = yf.download(
        symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        progress=False,
    )

    if df.empty:
        raise ValueError(f"No data returned for {ticker} ({symbol}).")

    df = df.reset_index()
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    available_columns = [col for col in expected_columns if col in df.columns]

    df = df[available_columns].copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Ticker"] = ticker
    df["Symbol"] = symbol

    return df

In [4]:
raw_frames = []

for ticker, symbol in MACRO_TICKERS.items():
    df_ticker = download_single_series(
        ticker=ticker,
        symbol=symbol,
        start_date=START_DATE,
        end_date=END_DATE,
        interval=INTERVAL,
    )
    raw_frames.append(df_ticker)

macro_raw_df = pd.concat(raw_frames, ignore_index=True)
macro_raw_df = macro_raw_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

macro_raw_df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Symbol
0,2017-01-03,102.870003,103.820000,102.589996,103.209999,103.209999,0,DXY,DX-Y.NYB
1,2017-01-04,103.180000,103.440002,102.389999,102.699997,102.699997,0,DXY,DX-Y.NYB
2,2017-01-05,102.430000,102.510002,101.300003,101.519997,101.519997,0,DXY,DX-Y.NYB
3,2017-01-06,101.419998,102.290001,101.370003,102.220001,102.220001,0,DXY,DX-Y.NYB
4,2017-01-09,102.260002,102.519997,101.800003,101.930000,101.930000,0,DXY,DX-Y.NYB


## Save raw combined dataset

The raw combined dataset is stored before further transformation so the original download remains available for inspection and reproducibility.

In [5]:
macro_raw_df.to_csv(MACRO_RAW_OUTPUT_PATH, index=False)
print(f"Saved raw macro dataset to: {MACRO_RAW_OUTPUT_PATH}")

Saved raw macro dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\raw\macro_long_raw.csv


## Basic inspection

In [6]:
print(macro_raw_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9270 entries, 0 to 9269
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       9270 non-null   datetime64[ns]
 1   Open       9270 non-null   float64       
 2   High       9270 non-null   float64       
 3   Low        9270 non-null   float64       
 4   Close      9270 non-null   float64       
 5   Adj Close  9270 non-null   float64       
 6   Volume     9270 non-null   int64         
 7   Ticker     9270 non-null   object        
 8   Symbol     9270 non-null   object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(2)
memory usage: 651.9+ KB
None


In [7]:
macro_raw_df.isna().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
Ticker       0
Symbol       0
dtype: int64

In [8]:
macro_raw_df.duplicated(subset=["Ticker", "Date"]).sum()

np.int64(0)

## Data quality summary by factor

This summary shows the available history and missing values for each selected macro factor.

In [9]:
summary_dict = {
    "symbol": ("Symbol", "first"),
    "first_date": ("Date", "min"),
    "last_date": ("Date", "max"),
    "n_rows": ("Date", "count"),
    "missing_open": ("Open", lambda s: s.isna().sum()),
    "missing_high": ("High", lambda s: s.isna().sum()),
    "missing_low": ("Low", lambda s: s.isna().sum()),
    "missing_close": ("Close", lambda s: s.isna().sum()),
    "missing_volume": ("Volume", lambda s: s.isna().sum()),
}

if "Adj Close" in macro_raw_df.columns:
    summary_dict["missing_adj_close"] = ("Adj Close", lambda s: s.isna().sum())

macro_quality_summary = (
    macro_raw_df
    .groupby("Ticker")
    .agg(**summary_dict)
    .reset_index()
)

macro_quality_summary

,Ticker,symbol,first_date,last_date,n_rows,missing_open,missing_high,missing_low,missing_close,missing_volume,missing_adj_close
0,DXY,DX-Y.NYB,2017-01-03,2026-03-23,2319,0,0,0,0,0,0
1,Gold,GC=F,2017-01-03,2026-03-23,2318,0,0,0,0,0,0
2,SP500,^GSPC,2017-01-03,2026-03-20,2316,0,0,0,0,0,0
3,VIX,^VIX,2017-01-03,2026-03-23,2317,0,0,0,0,0,0


In [10]:
macro_quality_summary.to_csv(MACRO_QUALITY_SUMMARY_OUTPUT_PATH, index=False)
print(f"Saved macro quality summary to: {MACRO_QUALITY_SUMMARY_OUTPUT_PATH}")

Saved macro quality summary to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\macro_data_quality_summary.csv


## Create wide-format close dataset

For later use, the macro series are stored in wide format with one column per factor.

In [11]:
macro_wide_full_df = (
    macro_raw_df
    .pivot(index="Date", columns="Ticker", values="Close")
    .sort_index()
)

macro_wide_full_df = macro_wide_full_df.dropna(how="all")

print("Macro full shape:", macro_wide_full_df.shape)
macro_wide_full_df.head()

Macro full shape: (2320, 4)


Ticker,DXY,Gold,SP500,VIX
Date,,,,
2017-01-03,103.209999,1160.400024,2257.830078,12.85
2017-01-04,102.699997,1163.800049,2270.750000,11.85
2017-01-05,101.519997,1179.699951,2269.000000,11.67
2017-01-06,102.220001,1171.900024,2276.979980,11.32
2017-01-09,101.930000,1183.500000,2268.899902,11.56


## Align macro data to the crypto modeling period

The aligned macro dataset is restricted to the date range of the aligned crypto dataset so that both can be merged later without additional date filtering.

In [12]:
crypto_aligned_close_df = pd.read_csv(
    CRYPTO_ALIGNED_CLOSE_INPUT_PATH,
    parse_dates=["Date"],
    index_col="Date",
)

print("Crypto aligned start date:", crypto_aligned_close_df.index.min())
print("Crypto aligned end date:", crypto_aligned_close_df.index.max())

Crypto aligned start date: 2020-04-10 00:00:00
Crypto aligned end date: 2026-03-23 00:00:00


In [13]:
macro_wide_aligned_df = macro_wide_full_df.loc[
    crypto_aligned_close_df.index.min():crypto_aligned_close_df.index.max()
].copy()

print("Macro aligned shape before reindex:", macro_wide_aligned_df.shape)

Macro aligned shape before reindex: (1497, 4)


## Reindex to the crypto aligned dates

The macro series may not naturally have exactly the same date index as the crypto dataset. To ensure compatibility, the macro data is reindexed to the aligned crypto dates.

In [14]:
macro_wide_aligned_df = macro_wide_aligned_df.reindex(crypto_aligned_close_df.index)

print("Macro aligned shape after reindex:", macro_wide_aligned_df.shape)
macro_wide_aligned_df.head()

Macro aligned shape after reindex: (2174, 4)


Ticker,DXY,Gold,SP500,VIX
Date,,,,
2020-04-10,NaN,NaN,NaN,NaN
2020-04-11,NaN,NaN,NaN,NaN
2020-04-12,NaN,NaN,NaN,NaN
2020-04-13,99.349998,1744.800049,2761.629883,41.169998
2020-04-14,98.889999,1756.699951,2846.060059,37.759998


## Missing values after alignment

Some missing values may appear because macro series and crypto series do not always share exactly the same calendar. This is expected and should be documented before later use.

In [15]:
print("Missing values in macro full dataset:")
print(macro_wide_full_df.isna().sum())

print()
print("Missing values in macro aligned dataset:")
print(macro_wide_aligned_df.isna().sum())

Missing values in macro full dataset:
Ticker
DXY      1
Gold     2
SP500    4
VIX      3
dtype: int64

Missing values in macro aligned dataset:
Ticker
DXY      678
Gold     678
SP500    681
VIX      680
dtype: int64


<div class="alert alert-block alert-info">
These missing values mainly occur on weekends and market holidays, since macro factors follow trading calendars while cryptocurrencies trade continuously.
</div>

## Save cleaned outputs

Two versions are stored:

- full macro dataset with the complete historical range
- aligned macro dataset restricted to the crypto modeling period

In [16]:
macro_wide_full_df.to_csv(MACRO_WIDE_FULL_OUTPUT_PATH, index=True)
macro_wide_aligned_df.to_csv(MACRO_WIDE_ALIGNED_OUTPUT_PATH, index=True)

print(f"Saved macro full dataset to: {MACRO_WIDE_FULL_OUTPUT_PATH}")
print(f"Saved macro aligned dataset to: {MACRO_WIDE_ALIGNED_OUTPUT_PATH}")

Saved macro full dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\macro_wide_close_full.csv
Saved macro aligned dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\macro_wide_close_aligned.csv
